In [ ]:
def main(datasources, start_date, end_date):
    """
    成交节奏因子（正交）：日内成交时间分布 + 脉冲特征；
    过热节奏 → 次日反转，factor 越大越好。
    """
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]
    LOOKBACK_DAYS = 7
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d %H:%M:%S")

    sql = f"""
    WITH cte_minute AS (
        SELECT
            date_trunc('day', date)::DATE::DATETIME AS date,
            instrument,
            volume, deal_number, close,
            (hour(date) * 60 + minute(date)) AS minute_of_day,
            lag(close) OVER (
                PARTITION BY instrument, date_trunc('day', date)::DATE ORDER BY date
            ) AS prev_close
        FROM {bar1m}
        WHERE ask_price1 > 0 AND bid_price1 > 0
    ),
    cte_feat AS (
        SELECT
            date, instrument, minute_of_day, volume, deal_number,
            volume / NULLIF(deal_number, 0) AS avg_trade_size,
            CASE
                WHEN prev_close IS NOT NULL AND prev_close > 0
                     AND close > prev_close AND volume > 0
                THEN 1.0 ELSE 0.0
            END AS up_vol_bar
        FROM cte_minute
    )
    SELECT
        date,
        instrument,
        nanstd(deal_number) / NULLIF(avg(deal_number), 0) AS trade_burst,
        sum(CASE WHEN minute_of_day >= 870 THEN volume ELSE 0 END)::DOUBLE
            / NULLIF(sum(CASE WHEN minute_of_day < 630 THEN volume ELSE 0 END), 0) AS pm_am_vol,
        sum(CASE WHEN minute_of_day >= 900 THEN volume ELSE 0 END)::DOUBLE
            / NULLIF(sum(volume), 0) AS close_rush,
        avg(avg_trade_size) AS avg_trade_size,
        avg(up_vol_bar) AS vol_sync
    FROM cte_feat
    GROUP BY date, instrument
    ORDER BY date, instrument
    """

    df = dai.query(sql, filters={"date": [query_start, end_date]}, compression=True).df()
    df = df.drop_duplicates(["date", "instrument"], keep="last")
    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    weights = {
        "trade_burst": 0.25,
        "pm_am_vol": 0.25,
        "close_rush": 0.22,
        "avg_trade_size": 0.16,
        "vol_sync": 0.12,
    }

    for col in weights:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df.groupby("date")[col].transform(
            lambda s: s.fillna(s.median()) if s.notna().any() else s
        )

    df["factor"] = 0.0
    for col, w in weights.items():
        pct = df.groupby("date")[col].rank(pct=True, method="average")
        df["factor"] += w * (1.0 - pct)

    df = df[(df["date"] >= pd.to_datetime(start_date)) & (df["date"] <= pd.to_datetime(end_date))]
    df = df.dropna(subset=["factor"])

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["instrument"] = stk_pool["instrument"].astype(str)
    stk_pool["date"] = pd.to_datetime(stk_pool["date"])

    out = pd.merge(df[["date", "instrument", "factor"]], stk_pool, how="inner", on=["date", "instrument"])
    out = out.drop_duplicates(["date", "instrument"], keep="last")
    out["factor"] = out["factor"].astype("float64")
    return out.sort_values(["date", "instrument"]).reset_index(drop=True)
